In [7]:
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import sys
sys.path.append('../')
from src.processing import process_data

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
submission = pd.read_csv('../data/sample_submission.csv')

train_processed = process_data(train, is_train=True)
test_processed = process_data(test, is_train=False)

X = train_processed.drop(columns=['exam_score'])
y = train_processed['exam_score']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'random_state': 42,
    'learning_rate': 0.01,      # 降低步长，增加精度
    'num_leaves': 31,           # 叶子节点数
    'feature_fraction': 0.8,    # 类似 XGB 的 colsample_bytree
    'bagging_fraction': 0.8,    # 类似 XGB 的 subsample
    'bagging_freq': 5,
    'max_depth': 8              # 限制深度防止过拟合
}

# 2. 创建模型实例
# n_estimators 设大一点，配合 early_stopping 使用
lgbm_model = lgb.LGBMRegressor(n_estimators=5000, **lgb_params)

# 3. 训练
print("开始训练 LightGBM...")
lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100), # 100轮不提升则停止
        lgb.log_evaluation(period=200)           # 每200轮打印一次日志
    ]
)

# 4. 预测与评估
lgbm_val_preds = lgbm_model.predict(X_val)
rmse_lgbm = np.sqrt(mean_squared_error(y_val, lgbm_val_preds))

print("-" * 30)
print(f"✨ LightGBM 验证集 RMSE: {rmse_lgbm:.4f}")
print("-" * 30)

开始训练 LightGBM...
Training until validation scores don't improve for 100 rounds
[200]	valid_0's rmse: 9.62049
[400]	valid_0's rmse: 8.92993
[600]	valid_0's rmse: 8.83798
[800]	valid_0's rmse: 8.81689
[1000]	valid_0's rmse: 8.80418
[1200]	valid_0's rmse: 8.79469
[1400]	valid_0's rmse: 8.78805
[1600]	valid_0's rmse: 8.78294
[1800]	valid_0's rmse: 8.77759
[2000]	valid_0's rmse: 8.77401
[2200]	valid_0's rmse: 8.7713
[2400]	valid_0's rmse: 8.76876
[2600]	valid_0's rmse: 8.76658
[2800]	valid_0's rmse: 8.7644
[3000]	valid_0's rmse: 8.76254
[3200]	valid_0's rmse: 8.7606
[3400]	valid_0's rmse: 8.75895
[3600]	valid_0's rmse: 8.7572
[3800]	valid_0's rmse: 8.75602
[4000]	valid_0's rmse: 8.75502
[4200]	valid_0's rmse: 8.75395
[4400]	valid_0's rmse: 8.75281
[4600]	valid_0's rmse: 8.75195
[4800]	valid_0's rmse: 8.75158
[5000]	valid_0's rmse: 8.75095
Did not meet early stopping. Best iteration is:
[5000]	valid_0's rmse: 8.75095
------------------------------
✨ LightGBM 验证集 RMSE: 8.7509
----------------

In [1]:
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd
import sys
import os

# 导入你的工程化函数
sys.path.append('../')
from src.processing import process_data
from src.utils import save_submission

# 1. 加载数据
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

# 2. 特征工程处理
train_processed = process_data(train, is_train=True)
test_processed = process_data(test, is_train=False)

# --- 重要：对齐测试集特征 ---
# 确保 X 和 X_test 拥有一模一样的列（数量和顺序）
features = [c for c in train_processed.columns if c != 'exam_score']
X = train_processed[features]
y = train_processed['exam_score']

# 对测试集补全缺失列并排序
for col in features:
    if col not in test_processed.columns:
        test_processed[col] = 0
X_test = test_processed[features]

# 3. 配置参数
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'random_state': 42,
    'learning_rate': 0.005,
    'num_leaves': 64,          # GPU 算力强，可以稍微增加叶子数捕捉更多细节
    'max_depth': 12,           # 配合 num_leaves 增加深度
    'feature_fraction': 0.7,   # 增加特征随机性，防止过拟合
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
}

# 4. 5-Fold 交叉验证训练循环
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))          # 存储验证集预测值
test_preds_total = np.zeros(len(X_test)) # 存储 5 个模型对测试集的平均预测

print(f"🚀 开始 5-Fold 交叉验证训练 (数据总量: {len(X)})...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    # 按照索引切分数据
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
    
    # 定义模型
    model = lgb.LGBMRegressor(n_estimators=10000, **lgb_params)
    
    # 训练 (使用回调函数控制停止和日志)
    model.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200), # 增加耐性，防止早停在局部最优
            lgb.log_evaluation(period=0)             # 静默模式，只打印最后的 Fold 结果
        ]
    )
    
    # 记录当前折的验证集预测
    fold_val_preds = model.predict(X_val_fold)
    oof_preds[val_idx] = fold_val_preds
    
    # 累加测试集预测（除以折数取平均）
    test_preds_total += model.predict(X_test) / kf.n_splits
    
    # 计算当前折分数
    fold_rmse = np.sqrt(mean_squared_error(y_val_fold, fold_val_preds))
    print(f"✅ Fold {fold+1} 完成! RMSE: {fold_rmse:.4f}")

# 5. 最终评估与保存
total_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print("\n" + "="*30)
print(f"🏆 5-Fold 平均验证集 RMSE: {total_rmse:.4f}")
print("="*30)

🚀 开始 5-Fold 交叉验证训练 (数据总量: 630000)...
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[9995]	valid_0's rmse: 8.73946
✅ Fold 1 完成! RMSE: 8.7395
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[10000]	valid_0's rmse: 8.74516
✅ Fold 2 完成! RMSE: 8.7452
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[10000]	valid_0's rmse: 8.7378
✅ Fold 3 完成! RMSE: 8.7378
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[10000]	valid_0's rmse: 8.7537
✅ Fold 4 完成! RMSE: 8.7537
Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[9970]	valid_0's rmse: 8.77224
✅ Fold 5 完成! RMSE: 8.7722

🏆 5-Fold 平均验证集 RMSE: 8.7497


In [2]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd
import sys
import os

# 导入你的工程化函数
sys.path.append('../')
from src.processing import process_data
from src.utils import save_submission

# 1. 加载数据
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

# 2. 特征工程处理
train_processed = process_data(train, is_train=True)
test_processed = process_data(test, is_train=False)

# --- 重要：对齐测试集特征 ---
# 确保 X 和 X_test 拥有一模一样的列（数量和顺序）
features = [c for c in train_processed.columns if c != 'exam_score']
X = train_processed[features]
y = train_processed['exam_score']

# 对测试集补全缺失列并排序
for col in features:
    if col not in test_processed.columns:
        test_processed[col] = 0
X_test = test_processed[features]

# 1. XGBoost GPU 参数设置
xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'learning_rate': 0.005,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'tree_method': 'hist',
    'device': 'cuda',
    # --- 在这里添加早停参数 ---
    'early_stopping_rounds': 200, 
    'n_estimators': 10000
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_xgb = np.zeros(len(X))
test_preds_xgb = np.zeros(len(X_test))

print("🚀 开始 XGBoost 5-Fold 训练...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train_f, X_val_f = X.iloc[train_idx], X.iloc[val_idx]
    y_train_f, y_val_f = y.iloc[train_idx], y.iloc[val_idx]
    
    model = xgb.XGBRegressor(**xgb_params)
    
    model.fit(
        X_train_f, y_train_f,
        eval_set=[(X_val_f, y_val_f)],
        verbose=0
    )
    
    oof_preds_xgb[val_idx] = model.predict(X_val_f)
    test_preds_xgb += model.predict(X_test) / kf.n_splits
    
    fold_rmse = np.sqrt(mean_squared_error(y_val_f, oof_preds_xgb[val_idx]))
    print(f"✅ Fold {fold+1} 完成! RMSE: {fold_rmse:.4f}")

total_rmse_xgb = np.sqrt(mean_squared_error(y, oof_preds_xgb))
print(f"\n🏆 XGBoost 5-Fold 平均 RMSE: {total_rmse_xgb:.4f}")

🚀 开始 XGBoost 5-Fold 训练...


D:\anaconda\envs\pytorch-env\lib\site-packages\xgboost\core.py:158: UserWarning: [22:15:17] WARNING: C:\b\abs_52v3kadn8m\croot\xgboost-split_1748343554494\work\src\context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)


✅ Fold 1 完成! RMSE: 8.7421


D:\anaconda\envs\pytorch-env\lib\site-packages\xgboost\core.py:158: UserWarning: [22:17:09] WARNING: C:\b\abs_52v3kadn8m\croot\xgboost-split_1748343554494\work\src\context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)


✅ Fold 2 完成! RMSE: 8.7473


...

✅ Fold 3 完成! RMSE: 8.7407


D:\anaconda\envs\pytorch-env\lib\site-packages\xgboost\core.py:158: UserWarning: [22:20:59] WARNING: C:\b\abs_52v3kadn8m\croot\xgboost-split_1748343554494\work\src\context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)


✅ Fold 4 完成! RMSE: 8.7574


D:\anaconda\envs\pytorch-env\lib\site-packages\xgboost\core.py:158: UserWarning: [22:22:54] WARNING: C:\b\abs_52v3kadn8m\croot\xgboost-split_1748343554494\work\src\context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)


✅ Fold 5 完成! RMSE: 8.7748

🏆 XGBoost 5-Fold 平均 RMSE: 8.7525


In [9]:
light_preds = lgbm_model.predict(test_processed)

In [10]:
final_blended_preds = (light_preds * 0.5) + (test_preds_xgb * 0.5)

In [11]:
print("正在融合 LGBM (8.7497) 和 XGBoost (8.7525)...")
from src.utils import save_submission
save_submission(final_blended_preds, test)

正在融合 LGBM (8.7497) 和 XGBoost (8.7525)...
🚀 提交文件已生成: ../submissions\sub_0108_2231.csv


'../submissions\\sub_0108_2231.csv'